# Exercise 9.3 — One toolbox for explanations: DALEX

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day9/day6-03-dalex.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. Cells with `______` are blanks for you to fill in.*

Exercises 9.1 and 9.2 used a different package for each explanation. The
**DALEX** package collects these methods, and more, behind one interface: you
wrap the fitted model once in an **explainer**, then ask the explainer for
whatever you want to know. The same package exists in R (with **DALEXtra**
for tidymodels) and in Python (`dalex`), with the same function names.

**Theory:** the explainer object; model performance; permutation importance
with a loss function; partial dependence and individual conditional
expectation (ceteris-paribus) profiles; break-down and SHAP explanations of one
prediction.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn "dalex>=1.8"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import dalex as dx             # explanations for any model

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, cohen_kappa_score

------------------------------------------------------------------------

### Part 1 — The model and the explainer

The black box is the random forest of Exercises 9.1 and 9.2: the same data
([`data/diabetes_clean.csv`](data/diabetes_clean.csv)), split and five predictors. The
predictors stay in their own units. A forest splits on one variable at a time,
so it does not need them scaled, and the plots below are easier to read in
mg/dL and years than in standard deviations.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv")
diabetes = diabetes[["pregnant", "glucose", "mass", "pedigree", "age", "diabetes"]]
diabetes["diabetes"] = (diabetes["diabetes"] == "pos").astype(int)       # 1 = pos, 0 = neg

is_test = diabetes.index % 5 == 0               # every fifth person is held back
train = diabetes[~is_test]
test  = diabetes[is_test]

predictors = ["pregnant", "glucose", "mass", "pedigree", "age"]

rf = RandomForestClassifier(n_estimators=500, random_state=1)
rf = rf.fit(train[predictors], train["diabetes"])

Evaluate it on the test set, with the predicted classes (threshold 0.5):

In [ ]:
test_class = rf.predict(test[predictors])          # predicted class, threshold 0.5

print("accuracy:", round(accuracy_score(test["diabetes"], test_class), 3))
print("kappa:   ", round(cohen_kappa_score(test["diabetes"], test_class), 3))

Accuracy 0.760 and Cohen's kappa 0.451 (kappa is accuracy corrected for
the agreement expected by chance). The R forest is grown from different
random numbers and gets 0.767 and 0.459.

Now wrap the model in an explainer. `dx.Explainer()` needs the model, the
predictors of the data to explain it on (the 150 test patients), and the true
outcome `y` **as numbers, 1 for diabetes and 0 for none**. Give the explainer
a label; it appears on every result.

In [ ]:
# Replace every ______ below, then run this cell.
explainer = dx.Explainer(
    rf,
    data=test[______],
    y=test["diabetes"],                           # already 1 = diabetes, 0 = none
    label="Random forest",
    verbose=False,                                # no progress report
)

> **Stuck?**
>
> `data` is the table of predictors only, without the outcome column: the same
> columns the forest was fitted on.

Ask the explainer how well the model performs:

In [ ]:
performance = explainer.model_performance()
performance.result.round(3)

The AUC, 0.824, is the same as `roc_auc_score()` gives on these
probabilities: a quick check that the explainer is using the probability of
diabetes. Recall, precision, F1 and accuracy use a threshold of 0.5.

> **Learning points**
>
> - An explainer wraps a fitted model with the data to explain it on and the true outcome; every explanation is then one function call.
> - The outcome must be numeric, 1 for the event and 0 otherwise; check the explainer's report and the AUC before going on.

------------------------------------------------------------------------

### Part 2 — Global: which variables matter?

`model_parts()` computes the **permutation importance** of Exercise 9.1:
shuffle one variable, measure how much worse the model gets, repeat
(10 times by default) and average. DALEX measures "worse" with a **loss**,
which by default for a classifier is $1 - \text{AUC}$, so a larger loss is a
worse model. Two reference rows come with the result: `_full_model_`, the loss
with nothing shuffled, and `_baseline_`, the loss when *every* variable is
shuffled at once (a model that knows nothing, AUC about 0.5).

In [ ]:
importance_dalex = explainer.model_parts(random_state=1)

importance_dalex.result[["variable", "dropout_loss"]].round(3)

In [ ]:
result = importance_dalex.result
full_loss = result.loc[result["variable"] == "_full_model_", "dropout_loss"].iloc[0]
bars = result[~result["variable"].isin(["_full_model_", "_baseline_"])]
bars = bars.sort_values("dropout_loss")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.barh(bars["variable"], bars["dropout_loss"] - full_loss, left=full_loss, color="coral")
ax.axvline(full_loss, color="black", linestyle="--")
ax.set_xlim(left=full_loss - 0.01)                # leave room to see the dashed line
ax.set_xlabel("1 - AUC after shuffling the variable (dashed: no variable shuffled)")
ax.set_title("Random forest: permutation importance (DALEX)")
fig.tight_layout()
plt.show()

# dalex's own plot gives the same picture, as an interactive plotly figure:
# importance_dalex.plot()

The full model has a loss of 0.176 (an AUC of 0.824). Shuffling glucose
raises it to 0.340; shuffling any other variable raises it by between
0.009 (pedigree) and 0.035 (BMI). This is the ranking of Exercise 9.1, Part 3, reached with
one line of code.

> **Learning points**
>
> - `model_parts()` is permutation importance; its loss for a classifier is 1 − AUC, so higher means the model got worse.
> - `_full_model_` and `_baseline_` frame the result: the loss with nothing shuffled, and with everything shuffled.

------------------------------------------------------------------------

### Part 3 — How does the prediction change with one variable?

An importance says *how much* a variable matters, not *how*. A
**partial dependence profile** shows how: set glucose to 60 for every test
patient, keeping everything else about them as it is, and average the
predicted probabilities; then set it to 61, and so on, up to 199. The curve of
these averages is what the model predicts, on average, at each glucose level.

One patient's curve on its own (every other variable held at her values) is
an **individual conditional expectation (ICE)** or **ceteris-paribus** profile
("all else being equal"). The partial dependence profile is the average of the
150 ICE profiles.

In [ ]:
pdp_glucose = explainer.model_profile(variables=["glucose"], type="partial",
                                      verbose=False)

pdp_glucose.result.iloc[::20].round(3)   # every 20th of the 101 grid points

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(pdp_glucose.result["_x_"], pdp_glucose.result["_yhat_"], color="steelblue", linewidth=2)
ax.set_xlabel("Glucose (mg/dL)")
ax.set_ylabel("Average predicted probability of diabetes")
ax.set_title("Partial dependence profile: glucose")
fig.tight_layout()
plt.show()

Up to a glucose of about 90 the curve stays between about 0.16 and 0.20. It
then climbs, gradually up to about 140 and steeply between 140 and 160, to
about 0.6, and stays between 0.6 and 0.66 above that. A logistic regression would have drawn a smooth S; the forest draws
steps, because it is made of splits.

Now the ICE profiles behind the average. They are stored in `raw_profiles`:

In [ ]:
ice = pdp_glucose.raw_profiles.result

fig, ax = plt.subplots(figsize=(6, 3.5))
for patient_id in ice["_ids_"].unique():
    one = ice[ice["_ids_"] == patient_id]
    ax.plot(one["glucose"], one["_yhat_"], color="grey", alpha=0.3, linewidth=0.7)
ax.plot(pdp_glucose.result["_x_"], pdp_glucose.result["_yhat_"], color="steelblue", linewidth=2.5)
ax.set_xlabel("Glucose (mg/dL)")
ax.set_ylabel("Predicted probability of diabetes")
ax.set_title("ICE profiles (grey) and their average (blue)")
fig.tight_layout()
plt.show()

**Question!** Do all the grey curves have the same shape as the blue one? What
would it mean if some went up steeply and others hardly moved?

They rise in roughly the same place but by very different amounts, and they
start at different heights. Where ICE curves have different shapes, the effect
of glucose depends on the patient's other values: an **interaction**. The
average curve hides that, which is why it is worth drawing both.

**Your turn.** Draw the partial dependence profiles for the pedigree function
and for age. `model_profile()` takes several variables at once.

In [ ]:
# Replace every ______ below, then run this cell.
pdp_more = explainer.model_profile(variables=[______, ______], type="partial",
                                   verbose=False)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), sharey=True)
for ax, name in zip(axes, ["pedigree", "age"]):
    one = pdp_more.result[pdp_more.result["_vname_"] == name]
    ax.plot(one["_x_"], one["_yhat_"], color="steelblue", linewidth=2)
    ax.set_xlabel(name)
axes[0].set_ylabel("Average predicted probability of diabetes")
fig.tight_layout()
plt.show()

> **Learning points**
>
> - A partial dependence profile is the average prediction as one variable is moved through its range, everything else about each patient held fixed.
> - ICE (ceteris-paribus) profiles are the individual curves behind that average; when their shapes differ, the variable interacts with others.
> - A forest's profiles are step-shaped, because the forest is made of splits.

------------------------------------------------------------------------

### Part 4 — One patient: break-down and SHAP

`predict_parts()` explains one prediction. Take the first test patient again,
the 28-year-old with a glucose of 194 from Exercise 9.2.

In [ ]:
new_observation = test[predictors].iloc[[0]]

print(new_observation)
print(explainer.predict(new_observation))     # her predicted probability

**SHAP**, as in Exercise 9.2. DALEX estimates the Shapley values by adding the
variables in 25 random orders and averaging each variable's contribution, with
the 150 test patients as the background:

In [ ]:
shap_patient = explainer.predict_parts(new_observation, type="shap", B=25,
                                       random_state=1)

shap_average = shap_patient.result[shap_patient.result["B"] == 0]   # B = 0 holds the average over the 25 orders
shap_average = shap_average[["variable", "contribution"]]
shap_average.sort_values("contribution", key=abs, ascending=False).round(3)

In [ ]:
ordered = shap_average.sort_values("contribution", key=abs)
colours = np.where(ordered["contribution"] > 0, "firebrick", "steelblue")

fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(ordered["variable"], ordered["contribution"], color=colours)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("SHAP contribution to the predicted probability")
ax.set_title("SHAP values for the first test patient (dalex)")
fig.tight_layout()
plt.show()

Glucose contributes +0.349, much as in Exercise 9.2 (+0.350
there). The numbers are not identical: the background is the test patients
rather than the training rows, and 25 random orders give an estimate, not the
exact value.

**Break-down** is the simplest way to share out a prediction: start from the
average prediction (the *intercept*), add the variables **one at a time** in a
fixed order, and record how much the average prediction moves at each step.
SHAP is the average of break-downs over many orders.

In [ ]:
# Replace every ______ below, then run this cell.
breakdown_patient = explainer.predict_parts(new_observation, type=______)

breakdown_patient.result[["variable", "contribution", "cumulative"]].round(3)

> **Stuck?**
>
> The type is written as a string, with an underscore: look at the SHAP cell
> above and at `help(explainer.predict_parts)`.

**Question!** In the break-down, which variable comes first, and why does the
order matter? Would age get the same contribution if it were added first?

The contribution of a variable depends on what is already known when it is
added: knowing that a patient is 28 means something different before and
after you know her glucose is 194. Break-down picks one order; SHAP averages
over many, which is why it is the safer summary when variables interact.

All of these explanations, and more, can be browsed together in an
interactive dashboard:

In [ ]:
# Optional, not run here: an interactive dashboard of everything above
# arena = dx.Arena()
# arena.push_model(explainer)
# arena.push_observations(test[predictors])
# arena.run_server()

> **Learning points**
>
> - `predict_parts()` explains one prediction; `type = "break_down"` adds the variables in one order, `type = "shap"` averages over many orders.
> - When variables interact, a break-down depends on the order; SHAP removes that dependence by averaging.
> - One explainer, four questions: how good is the model, which variables matter, how does the prediction change with a variable, and why did this patient get this prediction.

## Stretch

> **Stretch — explain the logistic regression too**
>
> Fit the logistic regression of Exercise 9.2 on the same five predictors, wrap
> it in a second explainer with the label `"Logistic regression"`, and pass both
> `model_parts()` results (or both `model_profile()` results) to one plot. Where
> do the two models' glucose profiles agree, and where not?

> **Stretch — more repetitions**
>
> `model_parts()` uses 10 shuffles per variable by default. Run it with `B = 50`
> (R) or `B=50` (Python), twice with different seeds. Does the order of
> pregnancies, age and BMI stay the same?